# 手撕 BPE (Byte Pair Encoding)

## 背景
BPE 从字符级开始，贪心合并最高频的相邻 byte pair，直到词表达到目标大小。
GPT-2/GPT-3 使用 BPE 分词。

## 考察点
- 训练：统计 pair 频率 → 合并最高频 pair → 更新词表
- 编码：贪心应用已学合并规则
- 编码/解码一致性

In [ ]:
from collections import Counter, defaultdict

class BPE:
    def __init__(self, num_merges: int = 100) -> None:
        self.num_merges = num_merges
        self.merges = []  # 按顺序存储合并规则

    def _get_pairs(self, word: str) -> list:
        return [(word[i], word[i+1]) for i in range(len(word) - 1)]

    def _count_pairs(self, corpus: dict) -> Counter:
        counts = Counter()
        for word, freq in corpus.items():
            for pair in self._get_pairs(word):
                counts[pair] += freq
        return counts

    def train(self, texts: list) -> None:
        # 初始化：每个词拆成字符 + </w> 结束符
        word_freqs = Counter()
        for text in texts:
            for word in text.split():
                word_freqs[tuple(list(word) + ['</w>'])] += 1
        for _ in range(self.num_merges):
            pair_counts = self._count_pairs(word_freqs)
            if not pair_counts:
                break
            best_pair = pair_counts.most_common(1)[0][0]
            self.merges.append(best_pair)
            # 合并
            new_word_freqs = Counter()
            for word, freq in word_freqs.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == best_pair:
                        new_word.append(word[i] + word[i+1])
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_word_freqs[tuple(new_word)] = freq
            word_freqs = new_word_freqs

    def encode(self, text: str) -> torch.Tensor:
        tokens = []
        for word in text.split():
            word = list(word) + ['</w>']
            for merge in self.merges:
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == merge:
                        new_word.append(word[i] + word[i+1])
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                word = new_word
            tokens.extend(word)
        return tokens

In [ ]:
# 验证 BPE
texts = ["low low low low low", "lower lower newer newer newer", "newest newest newest"]
bpe = BPE(num_merges=15)
bpe.train(texts)
print(f"学到的合并规则: {bpe.merges}")
encoded = bpe.encode("lower newest")
print(f"encode('lower newest'): {encoded}")
assert len(encoded) > 0, "编码结果非空"
assert all(isinstance(t, str) for t in encoded), "token 应为字符串"
print("✅ BPE 训练 + 编码验证通过")